# Fraud Detection — Logistic Regression from Scratch

---

**Repository:** Machine Learning from Scratch  
**Notebook:** 04 — Binary Classification with Logistic Regression  
**Algorithm:** Logistic Regression via Gradient Descent (no Scikit-Learn)  
**Scaling:** Manual Standardization — zero mean, unit variance  
**Author:** Ather-ops  

---

## Objective

Detect fraudulent transactions using three features — transaction amount, frequency, and location risk — by implementing Logistic Regression **completely from scratch** using only NumPy.

This is the first classification notebook in the from-scratch series. Every previous notebook predicted a continuous number. Here, the model predicts a **probability** and converts it to a binary class label.

---

## Regression vs Classification — The Core Shift

| Aspect | Linear Regression (Notebooks 01-03) | Logistic Regression (This Notebook) |
|--------|-------------------------------------|--------------------------------------|
| Output | Any real number | Probability in (0, 1) |
| Activation | None — raw linear output | Sigmoid function |
| Loss function | Mean Squared Error | Binary Cross-Entropy (Log Loss) |
| Gradient formula | `(2/n) * sum(error * X)` | `(1/n) * sum(X * (y_pred - Y))` |
| Final answer | The number itself | Threshold the probability to get 0 or 1 |
| Evaluation metrics | MSE, RMSE, MAE, R2 | Accuracy, Precision, Recall, FPR, Confusion Matrix |

---

## The Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Where $z = m_1 x_1 + m_2 x_2 + m_3 x_3 + b$ is the raw linear output.

The sigmoid squashes any value of $z$ into the range (0, 1):
- $z \to +\infty$ : $\sigma(z) \to 1$ — very likely fraud
- $z = 0$ : $\sigma(z) = 0.5$ — uncertain
- $z \to -\infty$ : $\sigma(z) \to 0$ — very likely legitimate

---

## Binary Cross-Entropy Loss

$$L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

When the true label is 1 (fraud), the loss is $-\log(\hat{p})$ — it is zero if the model predicts probability 1.0 and very large if the model predicts near 0. This is stricter than MSE and pushes the model to give confident correct predictions.

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Create dataset |
| 3 | Exploratory Data Analysis |
| 4 | Define helper functions — sigmoid, log loss, standardize |
| 5 | Standardize features |
| 6 | Initialize parameters |
| 7 | Gradient descent training loop |
| 8 | Loss curve visualization |
| 9 | Generate predictions and probabilities |
| 10 | Build confusion matrix from scratch |
| 11 | Compute evaluation metrics |
| 12 | Visualize results — confusion matrix heatmap and probability bars |
| 13 | Predict for a new transaction |

---
## Step 1 — Import Libraries

Same three libraries. No Scikit-Learn, no `confusion_matrix` import, no `roc_auc_score`. Everything is written manually.

| Library | Role |
|---------|------|
| `pandas` | Create and inspect the transaction DataFrame |
| `numpy` | Sigmoid, log loss, standardization, gradient math |
| `matplotlib.pyplot` | Loss curve, confusion matrix heatmap, probability visualization |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

---
## Step 2 — Create Dataset

We create a 10-transaction dataset inline — no CSV needed. Each row represents one bank transaction.

| Column | Type | Description |
|--------|------|-------------|
| `amount` | int | Transaction amount in currency units |
| `frequency` | int | How many transactions this account made recently |
| `location_risk` | int | 0 = familiar location, 1 = risky or unusual location |
| `fraud` | int | Target — 0 = legitimate, 1 = fraudulent |

**Pattern in the data:**  
High amount + high frequency + risky location = fraud. Low amount + low frequency + familiar location = legitimate. The model must learn this pattern from 10 examples.

**Why such a small dataset:**  
This notebook is about implementing the algorithm from scratch — every operation transparent and inspectable. A small dataset keeps the printed outputs readable and makes it easy to manually verify that each computation is correct.

In [ ]:
# ---------------- DATA ----------------
data = {
    "amount":       [500, 1200, 3000, 8000, 200, 15000, 400, 20000, 100, 25000],
    "frequency":    [1,2,3,4,1,6,1,7,1,8],
    "location_risk":[0,0,1,1,0,1,0,1,0,1],
    "fraud":        [0,0,0,1,0,1,0,1,0,1]
}

df = pd.DataFrame(data)

print("Transaction Dataset — 10 Observations")
print("=" * 70)
print(df.to_string(index=True))
print("=" * 70)
print(f"Total transactions  : {len(df)}")
print(f"Fraudulent (1)      : {df['fraud'].sum()}")
print(f"Legitimate (0)      : {(df['fraud'] == 0).sum()}")
print(f"Fraud rate          : {df['fraud'].mean() * 100:.1f}%")
print("=" * 70)

---
## Step 3 — Exploratory Data Analysis

We visualize the raw data to understand the separation between fraud and legitimate transactions before any modeling.

**What to look for:**

| Feature | Pattern that helps classification |
|---------|----------------------------------|
| `amount` | Fraud cases cluster at higher amounts |
| `frequency` | Fraud cases have higher transaction frequency |
| `location_risk` | Fraud cases have `location_risk = 1` |

If the two classes (fraud vs legitimate) are visually well separated on these features, logistic regression can find a good decision boundary. If they overlap heavily, a linear boundary may not be sufficient.

In [ ]:
# EDA — feature distributions by fraud label
fraud_df  = df[df["fraud"] == 1]
legit_df  = df[df["fraud"] == 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Amount
axes[0].scatter(legit_df.index, legit_df["amount"],
                color="steelblue", s=100, label="Legitimate", zorder=3)
axes[0].scatter(fraud_df.index, fraud_df["amount"],
                color="tomato", s=100, marker="^", label="Fraud", zorder=3)
axes[0].set_xlabel("Transaction Index")
axes[0].set_ylabel("Amount")
axes[0].set_title("Amount — Fraud vs Legitimate")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# Frequency
axes[1].scatter(legit_df.index, legit_df["frequency"],
                color="steelblue", s=100, label="Legitimate", zorder=3)
axes[1].scatter(fraud_df.index, fraud_df["frequency"],
                color="tomato", s=100, marker="^", label="Fraud", zorder=3)
axes[1].set_xlabel("Transaction Index")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Frequency — Fraud vs Legitimate")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

# Location risk bar
loc_counts = df.groupby(["location_risk", "fraud"]).size().unstack(fill_value=0)
loc_counts.plot(kind="bar", ax=axes[2], color=["steelblue", "tomato"],
                edgecolor="white", width=0.5)
axes[2].set_xlabel("Location Risk (0=Safe, 1=Risky)")
axes[2].set_ylabel("Count")
axes[2].set_title("Location Risk — Fraud Distribution")
axes[2].legend(["Legitimate", "Fraud"])
axes[2].set_xticklabels(["Safe", "Risky"], rotation=0)
axes[2].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.suptitle("EDA — Feature Distributions by Fraud Label",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("=" * 70)

---
## Step 4 — Helper Functions

We define three reusable functions before training. Writing them as named functions — not inline expressions — makes the training loop readable and the math auditable.

### `sigmoid(z)`

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Takes any real number $z$ (the linear combination of features and weights) and returns a probability between 0 and 1.

### `log_loss(y_true, y_pred)`

$$L = -\frac{1}{n} \sum \left[ y \log(\hat{p}) + (1-y) \log(1-\hat{p}) \right]$$

**Why `np.clip(y_pred, 1e-10, 1 - 1e-10)`:**  
`log(0)` is undefined — it returns `-inf`. If the model is very confident and wrong (predicts probability 0.0 for a fraud case), the loss would explode to infinity. Clipping forces all probabilities to stay in the range $[10^{-10}, 1 - 10^{-10}]$ — near zero and near one, but never exactly those values. This is a standard numerical stability trick used in all production ML frameworks.

### `standardize(X)`

$$z = \frac{X - \mu}{\sigma}$$

Transforms a feature to have mean = 0 and standard deviation = 1. More robust than divide-by-max (Notebook 03) because it handles the spread of the data, not just its maximum.

In [ ]:
# ---------------- FUNCTIONS ----------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    return -np.mean(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))

def standardize(X):
    return (X - np.mean(X)) / np.std(X)

# Verify sigmoid behavior
test_z = np.array([-10, -2, 0, 2, 10])
print("Sigmoid function verification:")
print("=" * 70)
for z_val in test_z:
    print(f"  sigmoid({z_val:+5.0f}) = {sigmoid(z_val):.6f}")
print("  (Should approach 0 for large negative z, 0.5 at z=0, 1.0 for large positive z)")
print("=" * 70)

---
## Step 5 — Standardize Features

We apply the `standardize` function to each of the three features.

**Why standardization is critical here:**

| Feature | Raw range | After standardization |
|---------|----------|----------------------|
| `amount` | 100 to 25000 | Mean ≈ 0, Std ≈ 1 |
| `frequency` | 1 to 8 | Mean ≈ 0, Std ≈ 1 |
| `location_risk` | 0 or 1 | Mean ≈ 0, Std ≈ 1 |

Without standardization, `amount` values of 25000 would produce enormous gradient magnitudes compared to `location_risk` values of 0 or 1. The gradient for `amount` would dominate every parameter update, and `location_risk` — a highly predictive binary feature — would barely influence the model.

After standardization, all three features contribute proportionally to the linear combination $z$, and gradient descent converges much faster.

In [ ]:
# ---------------- FEATURES ----------------
X1 = standardize(df["amount"].values)
X2 = standardize(df["frequency"].values)
X3 = standardize(df["location_risk"].values)
Y  = df["fraud"].values

print("Features After Standardization")
print("=" * 70)
for name, arr in [("X1 (amount)", X1), ("X2 (frequency)", X2), ("X3 (location_risk)", X3)]:
    print(f"  {name:22s} mean={arr.mean():+.6f}  std={arr.std():.6f}  range=[{arr.min():.3f}, {arr.max():.3f}]")
print("  (Mean should be 0.000000, std should be 1.000000)")
print("=" * 70)
print("Y (fraud labels):", Y)
print("=" * 70)

---
## Step 6 — Initialize Parameters

All four weights start at zero. The model begins with complete uncertainty — it predicts probability 0.5 for every transaction.

| Parameter | Initial | What it learns |
|-----------|---------|----------------|
| `m1` | 0.0 | How much standardized amount raises fraud probability |
| `m2` | 0.0 | How much standardized frequency raises fraud probability |
| `m3` | 0.0 | How much standardized location risk raises fraud probability |
| `b` | 0.0 | Base log-odds when all features are zero (at their mean) |

**Learning rate `lr = 0.1`:**  
Much larger than the regression notebooks (0.00001). This is possible because the features are standardized — all gradients are on a comparable scale. A larger learning rate means faster convergence per step.

**1500 epochs:**  
Log loss converges more slowly than MSE because the sigmoid gradient approaches zero for very confident predictions. More epochs ensure the weights reach their optimal values.

In [ ]:
# ---------------- MODEL INIT ----------------
m1 = m2 = m3 = b = 0.0
lr     = 0.1
epochs = 1500
n      = len(Y)

loss_history = []

print("Model Initialized")
print("=" * 70)
print(f"m1 = m2 = m3 = b = 0.0")
print(f"Learning rate (lr) : {lr}")
print(f"Epochs             : {epochs}")
print(f"Samples (n)        : {n}")
print("=" * 70)

# Initial prediction before any training
z_init = m1*X1 + m2*X2 + m3*X3 + b
p_init = sigmoid(z_init)
init_loss = log_loss(Y, p_init)
print(f"Initial probability for all transactions: {p_init[0]:.4f}  (all equal 0.5 — no knowledge)")
print(f"Initial log loss                        : {init_loss:.4f}")
print("=" * 70)

---
## Step 7 — Gradient Descent Training Loop

The training loop for logistic regression follows the same structure as the regression loops — but with two key differences:

1. The forward pass passes through `sigmoid()` before computing loss
2. The loss function is `log_loss()` instead of MSE

**Training loop structure:**

```
for each epoch:
    z      = m1*X1 + m2*X2 + m3*X3 + b      <- linear combination
    y_pred = sigmoid(z)                       <- convert to probability
    loss   = log_loss(Y, y_pred)              <- binary cross-entropy
    dm1    = (1/n) * sum(X1 * (y_pred - Y))  <- gradient for m1
    dm2    = (1/n) * sum(X2 * (y_pred - Y))  <- gradient for m2
    dm3    = (1/n) * sum(X3 * (y_pred - Y))  <- gradient for m3
    db     = (1/n) * sum(y_pred - Y)          <- gradient for b
    m1 -= lr * dm1                            <- update
    m2 -= lr * dm2
    m3 -= lr * dm3
    b  -= lr * db
```

**Why the gradient formula is `(1/n) * sum(X * (y_pred - Y))`:**

When you take the derivative of binary cross-entropy loss with respect to the weights, the sigmoid and log derivatives cancel elegantly:

$$\frac{\partial L}{\partial m_1} = \frac{1}{n} \sum (\hat{p}_i - y_i) \cdot x_{1i}$$

This is a clean result — the gradient looks almost identical to the MSE gradient, but the error term is now $(\hat{p} - Y)$ where $\hat{p}$ is a sigmoid probability, not a raw prediction.

In [ ]:
# ---------------- TRAINING ----------------
for epoch in range(epochs):
    z      = m1*X1 + m2*X2 + m3*X3 + b
    y_pred = sigmoid(z)

    loss = log_loss(Y, y_pred)
    loss_history.append(loss)

    dm1 = (1/n) * np.sum(X1 * (y_pred - Y))
    dm2 = (1/n) * np.sum(X2 * (y_pred - Y))
    dm3 = (1/n) * np.sum(X3 * (y_pred - Y))
    db  = (1/n) * np.sum(y_pred - Y)

    m1 -= lr * dm1
    m2 -= lr * dm2
    m3 -= lr * dm3
    b  -= lr * db

    if epoch % 200 == 0:
        print(f"Epoch {epoch} | Log Loss: {loss:.4f}")

---
## Step 8 — Loss Curve and Final Parameters

We plot the log loss across all 1500 epochs and print the final learned weights.

**Reading the final weights:**

Because features are standardized, the weights are directly comparable:
- The feature with the **largest positive weight** has the strongest influence on predicting fraud
- A **negative weight** would mean that feature is associated with lower fraud probability

Expected: `m1` (amount) and `m2` (frequency) should be strongly positive — high amounts and high frequency are classic fraud indicators. `m3` (location_risk) should also be positive — risky locations correlate with fraud.

In [ ]:
# ---------------- LOSS CURVE ----------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(loss_history, color="steelblue", linewidth=1.5)
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Log Loss")
axes[0].set_title("Fraud Detection Training Curve — All 1500 Epochs")
axes[0].grid(True)

axes[1].plot(loss_history[:300], color="tomato", linewidth=1.5)
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Log Loss")
axes[1].set_title("First 300 Epochs — Initial Drop")
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("=" * 70)
print(f"Initial log loss  : {loss_history[0]:.4f}")
print(f"Loss at epoch 300 : {loss_history[300]:.4f}")
print(f"Final log loss    : {loss_history[-1]:.4f}")
reduction = (1 - loss_history[-1] / loss_history[0]) * 100
print(f"Loss reduction    : {reduction:.2f}%")
print("=" * 70)
print("Final Learned Parameters:")
print(f"  m1 (amount)        : {m1:.4f}")
print(f"  m2 (frequency)     : {m2:.4f}")
print(f"  m3 (location_risk) : {m3:.4f}")
print(f"  b  (bias)          : {b:.4f}")
print("=" * 70)
ranked = sorted([("amount", abs(m1)), ("frequency", abs(m2)), ("location_risk", abs(m3))],
                key=lambda x: x[1], reverse=True)
print("Feature influence ranking:")
for r, (feat, w) in enumerate(ranked, 1):
    print(f"  {r}. {feat:15s} |weight| = {w:.4f}")
print("=" * 70)

---
## Step 9 — Predictions and Probabilities

We generate predicted probabilities for every transaction and apply a decision threshold to produce class labels.

**Threshold = 0.6:**  
The default threshold for logistic regression is 0.5. Using 0.6 here means the model must be at least 60% confident before flagging a transaction as fraud.

**Why adjust the threshold in fraud detection:**

| Threshold | Effect | Use case |
|-----------|--------|----------|
| Lower (e.g. 0.3) | More fraud flags — catches more fraud but more false alarms | When missing fraud is very costly |
| 0.5 | Balanced | General purpose |
| Higher (e.g. 0.7) | Fewer fraud flags — more confident before flagging | When false alarms are costly (customer complaints) |

The choice of threshold is a **business decision**, not just a technical one. Different thresholds produce different Precision-Recall tradeoffs.

In [ ]:
# ---------------- PREDICTION ----------------
z             = m1*X1 + m2*X2 + m3*X3 + b
probabilities = sigmoid(z)

threshold       = 0.6
predicted_class = (probabilities >= threshold).astype(int)

# Full prediction table
pred_df = pd.DataFrame({
    "Amount"     : df["amount"].values,
    "Frequency"  : df["frequency"].values,
    "Loc_Risk"   : df["location_risk"].values,
    "Actual"     : Y,
    "P(Fraud)"   : probabilities.round(4),
    "Predicted"  : predicted_class,
    "Correct"    : (Y == predicted_class).astype(int)
})

print(f"Predictions with threshold = {threshold}")
print("=" * 80)
print(pred_df.to_string(index=False))
print("=" * 80)
print(f"Correct predictions : {pred_df['Correct'].sum()} / {len(pred_df)}")
print("=" * 80)

---
## Step 10 — Confusion Matrix — Built from Scratch

We count TP, TN, FP, FN manually using a loop — no `sklearn.metrics.confusion_matrix`.

**Definitions:**

| Term | Full name | Meaning |
|------|-----------|--------|
| TP | True Positive | Predicted fraud, actual fraud — correct alert |
| TN | True Negative | Predicted legitimate, actual legitimate — correct clearance |
| FP | False Positive | Predicted fraud, actual legitimate — false alarm |
| FN | False Negative | Predicted legitimate, actual fraud — **missed fraud** |

**In fraud detection, False Negatives are the most dangerous.** A missed fraud (FN) means a real fraudulent transaction was cleared. A false alarm (FP) means a legitimate transaction was flagged — annoying but recoverable. The threshold should be tuned to minimize FN at acceptable FP levels.

In [ ]:
# ---------------- CONFUSION MATRIX ----------------
TP = TN = FP = FN = 0
for actual, pred in zip(Y, predicted_class):
    if   actual == 1 and pred == 1: TP += 1
    elif actual == 0 and pred == 0: TN += 1
    elif actual == 0 and pred == 1: FP += 1
    elif actual == 1 and pred == 0: FN += 1

print("Confusion Matrix")
print(f"TN:{TN} FP:{FP}")
print(f"FN:{FN} TP:{TP}")
print("=" * 70)
print(f"True  Positives (TP) : {TP}  — fraud correctly flagged")
print(f"True  Negatives (TN) : {TN}  — legitimate correctly cleared")
print(f"False Positives (FP) : {FP}  — false alarm")
print(f"False Negatives (FN) : {FN}  — missed fraud")
print("=" * 70)

---
## Step 11 — Evaluation Metrics — All Computed from Scratch

We compute four metrics manually from the TP, TN, FP, FN counts:

| Metric | Formula | What it measures |
|--------|---------|------------------|
| Accuracy | $\frac{TP + TN}{TP + TN + FP + FN}$ | Overall fraction of correct predictions |
| Precision | $\frac{TP}{TP + FP}$ | Of all fraud alerts, what fraction was actually fraud? |
| Recall (TPR) | $\frac{TP}{TP + FN}$ | Of all actual fraud cases, what fraction did the model catch? |
| FPR | $\frac{FP}{FP + TN}$ | Of all legitimate transactions, what fraction was wrongly flagged? |

**The Precision-Recall tradeoff:**  
Increasing the threshold increases Precision (fewer false alarms) but decreases Recall (more missed fraud). Decreasing the threshold does the opposite. There is no perfect setting — it depends entirely on the business cost of each error type.

**F1 Score — the harmonic mean:**

$$F1 = \frac{2 \times Precision \times Recall}{Precision + Recall}$$

F1 is the standard single metric for imbalanced classification problems. It balances Precision and Recall and avoids the pitfall of accuracy being misleading when one class dominates.

In [ ]:
# ---------------- METRICS ----------------
accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
fpr       = FP / (FP + TN) if (FP + TN) > 0 else 0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\nMetrics")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("FPR      :", fpr)
print("=" * 70)
print(f"Accuracy  : {accuracy:.4f}  ({accuracy*100:.1f}% of all transactions correctly classified)")
print(f"Precision : {precision:.4f}  ({precision*100:.1f}% of fraud alerts were real fraud)")
print(f"Recall    : {recall:.4f}  ({recall*100:.1f}% of actual fraud cases were caught)")
print(f"FPR       : {fpr:.4f}  ({fpr*100:.1f}% of legitimate transactions were wrongly flagged)")
print(f"F1 Score  : {f1:.4f}")
print("=" * 70)

---
## Step 12 — Visualization

Three plots to summarize model performance visually:

1. **Confusion matrix heatmap** — colored grid showing TP/TN/FP/FN counts
2. **Probability bar chart** — predicted fraud probability for every transaction, colored by true label
3. **Decision boundary line** — the 0.6 threshold visualized on the probability chart

The probability chart is particularly useful — it shows not just whether the model got each prediction right, but **how confident it was**. A bar near 0.9 for a fraud case is much better than a bar at 0.61 (just barely above the threshold).

In [ ]:
# Confusion matrix heatmap + probability bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix heatmap
cm_array = np.array([[TN, FP], [FN, TP]])
im = axes[0].imshow(cm_array, cmap="Blues")
plt.colorbar(im, ax=axes[0])
cell_labels = [["TN", "FP"], ["FN", "TP"]]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i,
                     f"{cell_labels[i][j]}\n{cm_array[i,j]}",
                     ha="center", va="center", fontsize=13, fontweight="bold",
                     color="white" if cm_array[i,j] > cm_array.max() / 2 else "black")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Predicted\nLegitimate", "Predicted\nFraud"], fontsize=10)
axes[0].set_yticklabels(["Actual\nLegitimate", "Actual\nFraud"], fontsize=10)
axes[0].set_title(f"Confusion Matrix (threshold={threshold})", fontsize=12)

# Probability bars
bar_colors = ["tomato" if y == 1 else "steelblue" for y in Y]
bars = axes[1].bar(range(len(probabilities)), probabilities,
                   color=bar_colors, edgecolor="white", width=0.6)
axes[1].axhline(threshold, color="black", linestyle="--",
                linewidth=1.5, label=f"Threshold = {threshold}")
for i, (bar, prob) in enumerate(zip(bars, probabilities)):
    axes[1].text(i, prob + 0.02, f"{prob:.2f}",
                 ha="center", fontsize=8)
axes[1].set_xticks(range(len(probabilities)))
axes[1].set_xticklabels([f"T{i}" for i in range(len(probabilities))], fontsize=9)
axes[1].set_xlabel("Transaction")
axes[1].set_ylabel("P(Fraud)")
axes[1].set_ylim(0, 1.2)
axes[1].set_title("Predicted Fraud Probabilities\n(Red=Actual Fraud, Blue=Actual Legitimate)")
axes[1].legend(fontsize=10)
axes[1].grid(True, axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("=" * 70)

---
## Step 13 — Predict a New Transaction

We use the trained model to assess the fraud risk of a completely new transaction.

**New transaction:**

| Feature | Raw Value | Context |
|---------|----------|---------|
| amount | 12000 | High value |
| frequency | 5 | Multiple recent transactions |
| location_risk | 1 | Risky location |

**Pipeline for new input:**
1. Apply the same `standardize()` logic — subtract training mean, divide by training std
2. Compute $z = m_1 x_1 + m_2 x_2 + m_3 x_3 + b$
3. Pass through `sigmoid(z)` to get fraud probability
4. Apply threshold to get the final binary decision

In [ ]:
# New transaction prediction
new_amount    = 12000
new_freq      = 5
new_loc_risk  = 1

# Standardize using training set statistics
new_x1 = (new_amount   - df["amount"].mean())       / df["amount"].std()
new_x2 = (new_freq     - df["frequency"].mean())    / df["frequency"].std()
new_x3 = (new_loc_risk - df["location_risk"].mean())/ df["location_risk"].std()

new_z     = m1*new_x1 + m2*new_x2 + m3*new_x3 + b
new_prob  = sigmoid(new_z)
new_class = int(new_prob >= threshold)
label     = "FRAUD" if new_class == 1 else "LEGITIMATE"

print("New Transaction Assessment")
print("=" * 70)
print(f"Amount           : {new_amount}")
print(f"Frequency        : {new_freq}")
print(f"Location Risk    : {new_loc_risk}")
print("=" * 70)
print(f"Standardized x1  : {new_x1:.4f}")
print(f"Standardized x2  : {new_x2:.4f}")
print(f"Standardized x3  : {new_x3:.4f}")
print("=" * 70)
print(f"Linear output z  : {new_z:.4f}")
print(f"Fraud probability: {new_prob:.4f}  ({new_prob*100:.1f}%)")
print(f"Threshold        : {threshold}")
print(f"Decision         : {label}")
print("=" * 70)

---
## Pipeline Summary

| Step | Action | Key concept |
|------|--------|-------------|
| 1 | Import numpy, pandas, matplotlib | No Scikit-Learn |
| 2 | Create 10-transaction dataset inline | 5 fraud, 5 legitimate |
| 3 | EDA — scatter by fraud label + location bar | Class separation visualized |
| 4 | Define sigmoid, log_loss, standardize | Three reusable functions |
| 5 | Standardize X1, X2, X3 | Mean=0, Std=1 for each feature |
| 6 | Initialize m1=m2=m3=b=0, lr=0.1 | Initial prediction is 0.5 for all |
| 7 | 1500-epoch gradient descent loop | Forward pass through sigmoid, log_loss, gradient, update |
| 8 | Loss curve + final parameters | Feature importance by weight magnitude |
| 9 | Predictions + probability table | Threshold 0.6 applied |
| 10 | Confusion matrix — manual loop | TP, TN, FP, FN counted by hand |
| 11 | Accuracy, Precision, Recall, FPR, F1 | All formulas written from scratch |
| 12 | Heatmap + probability bar chart | Visual model summary |
| 13 | New transaction prediction | Standardize with training stats, then predict |

---

## What This Notebook Proves

The Scikit-Learn classification notebook uses:
```python
model = LogisticRegression()
model.fit(X_train, Y_train)
y_pred = model.predict(X_test)
```

This notebook shows what those three lines actually do:
- 1500 forward passes through a sigmoid function
- 1500 log loss computations
- 6000 gradient computations (one per parameter per epoch)
- 6000 parameter updates
- A confusion matrix built by iterating through predictions manually
- Every metric derived from TP, TN, FP, FN counts

---

**Next notebook:** `03_Loan_approval_project.ipynb` —  from scratch,